# 08: Certificate Workflows and Surgical Decisions

This notebook demonstrates:
- decision-ready certificate inputs,
- explicit homeomorphism witnesses (`explicit_map`),
- how a non-zero obstruction forces `surgery_required`,
- how to obtain a surgery planning statement with `surgery_to_remove_impediments`.


In [1]:
import numpy as np
import scipy.sparse as sp

from pysurgery import (
    ChainComplex,
    FundamentalGroup,
    IntersectionForm,
    ObstructionResult,
    WhiteheadGroup,
    analyze_homeomorphism_3d,
    analyze_homeomorphism_4d,
    analyze_homeomorphism_high_dim,
    analyze_homeomorphism_high_dim_result,
    build_homeomorphism_witness,
    ThreeManifoldRecognitionCertificate,
    HomotopyCompletionCertificate,
    ProductAssemblyCertificate,
    DefiniteLatticeIsometryCertificate,
    surgery_to_remove_impediments,
)


## 1) Certificate objects


In [2]:
recognition_certificate = ThreeManifoldRecognitionCertificate(
    provided=True, source="example", exact=True, validated=True
)

homotopy_completion_certificate = HomotopyCompletionCertificate(
    provided=True, source="example", exact=True, validated=True
)

product_assembly_certificate = ProductAssemblyCertificate(
    provided=True, source="example", exact=True, validated=True
)

definite_lattice_isometry_certificate = DefiniteLatticeIsometryCertificate(
    provided=True, source="example", exact=True, validated=True,
    isometry_matrix=[[3, 2], [2, 1]]
)

(
    recognition_certificate.decision_ready(),
    homotopy_completion_certificate.decision_ready(),
    product_assembly_certificate.decision_ready(),
    definite_lattice_isometry_certificate.decision_ready(),
)


(True, True, True, True)

## 2) Explicit homeomorphism witness (`explicit_map`) in 4D


In [4]:
q1 = np.array([[1, 0], [0, 1]], dtype=np.int64)
u = np.array([[3, 2], [2, 1]], dtype=np.int64)
q2 = u.T @ q1 @ u

form1 = IntersectionForm(matrix=q1, dimension=4)
form2 = IntersectionForm(matrix=q2, dimension=4)

witness_res = build_homeomorphism_witness(
    m1=form1,
    m2=form2,
    ks1=0,
    ks2=0,
    simply_connected=True,
    definite_lattice_isometry_certificate={
        "provided": True,
        "source": "tutorial",
        "exact": True,
        "validated": True,
        "isometry_matrix": u.tolist(),
    },
)

print(witness_res.status)
print("explicit_map shape:", None if witness_res.witness is None else np.asarray(witness_res.witness.explicit_map).shape)
print("explicit_map: ", witness_res.witness.explicit_map)


success
explicit_map shape: (2, 2)
explicit_map:  [[-3 -2]
 [-2 -1]]


## 3) 3D recognition certificate usage


In [5]:
d1 = sp.csr_matrix(np.zeros((1, 1), dtype=np.int64))
d2 = sp.csr_matrix(np.array([[3]], dtype=np.int64))
d3 = sp.csr_matrix(np.zeros((1, 1), dtype=np.int64))
c3 = ChainComplex(
    boundaries={1: d1, 2: d2, 3: d3},
    dimensions=[0, 1, 2, 3],
    cells={0: 1, 1: 1, 2: 1, 3: 1},
)
pi = FundamentalGroup(generators=["g"], relations=[["g", "g", "g"]])

ok3, msg3 = analyze_homeomorphism_3d(
    c3, c3,
    recognition_certificate={
        "provided": True,
        "source": "tutorial",
        "exact": True,
        "validated": True,
    },
)
print(ok3, msg3[:120])


True SUCCESS: Homology/fundamental-group compatibility checks pass and a decision-ready 3-manifold recognition certificate co


## 4) Obstruction interference: `surgery_required`


In [6]:
d1h = sp.csr_matrix(np.zeros((1, 0), dtype=np.int64))
d5h = sp.csr_matrix(np.zeros((0, 1), dtype=np.int64))
c5 = ChainComplex(
    boundaries={1: d1h, 5: d5h},
    dimensions=[0, 1, 2, 3, 4, 5],
    cells={0: 1, 1: 0, 2: 0, 3: 0, 4: 0, 5: 1},
)

wh = WhiteheadGroup(rank=0, description="Wh(1)=0", computable=True, exact=True)
wall_obstruction = ObstructionResult(
    dimension=5,
    pi="1",
    computable=True,
    exact=True,
    value=1,
    modulus=None,
    assumptions=[],
)

high_res = analyze_homeomorphism_high_dim_result(
    c5, c5,
    dim=5,
    pi1=FundamentalGroup(generators=[], relations=[]),
    whitehead_group=wh,
    wall_obstruction=wall_obstruction,
)

print(high_res.status)  # expected: surgery_required
print(high_res.reasoning)


surgery_required
SURGERY_REQUIRED: Non-zero Wall obstruction detected in L_5(1) (value=1).


## 5) Surgery planning helper


In [7]:
can_remove, plan = surgery_to_remove_impediments(form1, target_sig=5)
print(can_remove)
print(plan)


True
PLAN: Connected sum with 3 copies of CP^2 changes signature by 3. This only addresses signature-level obstructions; complete homeomorphism analysis may still require Kirby-Siebenmann agreement, Whitehead-torsion vanishing, and Wall L-group obstruction checks.


## 6) High-dimensional API surface (certificate keys)

This cell keeps `analyze_homeomorphism_high_dim`, `homotopy_completion_certificate`, and
`product_assembly_certificate` usage visible in one place for coverage validation.


In [8]:
ok_high, msg_high = analyze_homeomorphism_high_dim(
    c5, c5,
    dim=5,
    pi_group="Z_2 x Z_3",
    whitehead_group=WhiteheadGroup(rank=0, description="Wh(Z_2 x Z_3)=0", computable=True, exact=True),
    wall_obstruction=ObstructionResult(
        dimension=5,
        pi="Z_2 x Z_3",
        computable=True,
        exact=True,
        value=0,
        modulus=None,
        assumptions=[],
        decomposition_kind="factor_surrogate",
        assembly_certified=False,
        obstructs=False,
        zero_certified=True,
    ),
    homotopy_completion_certificate={"provided": True, "source": "tutorial", "exact": True, "validated": True},
    product_assembly_certificate={"provided": True, "source": "tutorial", "exact": True, "validated": True},
)
print(ok_high, msg_high[:120])


True SUCCESS: Homology equivalence, Wh(pi_1)=0, vanishing Wall obstruction, and an exact validated homotopy-completion certif
